# Experimentos MLflow — treino e ajuste de hiperparâmetros (notebook)

Este notebook é o **local explícito** onde cada algoritmo é **treinado**, seus **hiperparâmetros são alterados a cada rodada** e o resultado é **registrado no MLflow**.

> **Kernel:** use o Python do ambiente virtual (`.venv`) — ele contém `mlflow` e `sklearn`. No terminal: `source .venv/bin/activate` antes de abrir o Jupyter, ou selecione o kernel `.venv` no VS Code/Cursor.

## O que existe no MLflow (dois experimentos)

| Experimento | Origem | Conteúdo |
|-------------|--------|----------|
| `evasao_escolar_escola_ano` | `run_educational_ml_suite()` | Run pai + 3 comparações (HGB, árvore, KNN) + modelo final ajustado |
| **`evasao_treino_parametros`** | **este notebook** | **30 training runs** — 10 rodadas por algoritmo, hiperparâmetros diferentes |

## Algoritmos treinados neste notebook

1. **HistGradientBoostingRegressor** — hiperparâmetros: `learning_rate`, `max_depth`, `max_iter`, `min_samples_leaf`, `l2_regularization`, `max_leaf_nodes`
2. **DecisionTreeRegressor** — hiperparâmetros: `max_depth`, `min_samples_leaf`
3. **KNeighborsRegressor** — hiperparâmetros: `n_neighbors`, `weights`

**Alvo:** `taxa_abandono_em` · **Validação:** holdout temporal (treino ≤ 2017, teste ≥ 2018)

**UI:** `bash scripts/mlflow_ui.sh` → http://localhost:5000

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "ml").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ml.baseline_municipio import evaluate_regression
from ml.mlflow_experiments import (
    EXPERIMENT_PLAN,
    MLFLOW_TUNING_EXPERIMENT,
    algorithm_display_name,
    build_pipeline_from_config,
    log_training_run_to_mlflow,
    prepare_split,
)
from ml.mlflow_tracking import configure_mlflow, default_tracking_uri

print("Experimento deste notebook:", MLFLOW_TUNING_EXPERIMENT)
print("Tracking URI:", default_tracking_uri())

Experimento deste notebook: evasao_treino_parametros
Tracking URI: sqlite:////home/fatima/Henrique_projetos/projeto_evasao_escolar/mlruns/mlflow.db


## Plano de experimentos — 10 rodadas por algoritmo (30 no total)

O plano está definido em `ml/mlflow_experiments.py` (`EXPERIMENT_PLAN`) e é exibido abaixo. Cada linha declara o **algoritmo**, os **hiperparâmetros** da rodada e o **run_name** no MLflow. O loop de treino usa esta lista e imprime os valores a cada iteração.

In [2]:
n_hgb = sum(1 for r in EXPERIMENT_PLAN if r["algorithm"] == "HistGradientBoosting")
n_tree = sum(1 for r in EXPERIMENT_PLAN if r["algorithm"] == "DecisionTree")
n_knn = sum(1 for r in EXPERIMENT_PLAN if r["algorithm"] == "KNeighbors")
print(
    f"Rodadas: {len(EXPERIMENT_PLAN)} total "
    f"(HGB={n_hgb}, Árvore={n_tree}, KNN={n_knn})"
)

plan_rows = []
for row in EXPERIMENT_PLAN:
    plan_rows.append(
        {
            "run_name": row["run_name"],
            "algorithm": row["algorithm"],
            "modelo_sklearn": algorithm_display_name(row["algorithm"]),
            **row["params"],
            "nota": row.get("nota", ""),
        }
    )
plan_df = pd.DataFrame(plan_rows)
display(plan_df)

Rodadas: 30 total (HGB=10, Árvore=10, KNN=10)


,run_name,algorithm,modelo_sklearn,learning_rate,max_depth,max_iter,min_samples_leaf,l2_regularization,max_leaf_nodes,nota,n_neighbors,weights
0,hgb_01_baseline,HistGradientBoosting,HistGradientBoostingRegressor,0.15,3.0,80.0,24.0,0.00,15.0,"HGB baseline — poucas iterações, folhas grandes.",NaN,NaN
1,hgb_02_mais_iter_l2,HistGradientBoosting,HistGradientBoostingRegressor,0.10,4.0,180.0,16.0,0.10,31.0,HGB — mais iterações + regularização L2 leve.,NaN,NaN
2,hgb_03_arvore_rasa,HistGradientBoosting,HistGradientBoostingRegressor,0.10,2.0,220.0,20.0,0.05,7.0,HGB — boosting com árvores rasas (max_depth=2).,NaN,NaN
3,hgb_04_balanceado,HistGradientBoosting,HistGradientBoostingRegressor,0.10,4.0,220.0,12.0,0.30,15.0,HGB — combinação balanceada (candidato a model...,NaN,NaN
4,hgb_05_lr_alto_curto,HistGradientBoosting,HistGradientBoostingRegressor,0.20,3.0,60.0,30.0,0.00,15.0,"HGB — learning rate alto, poucas iterações.",NaN,NaN
5,hgb_06_lr_baixo_longo,HistGradientBoosting,HistGradientBoostingRegressor,0.05,3.0,300.0,18.0,0.10,31.0,"HGB — learning rate baixo, treino longo.",NaN,NaN
6,hgb_07_profundo_moderado,HistGradientBoosting,HistGradientBoostingRegressor,0.08,5.0,200.0,14.0,0.15,31.0,HGB — árvores mais profundas (max_depth=5).,NaN,NaN
7,hgb_08_folhas_pequenas,HistGradientBoosting,HistGradientBoostingRegressor,0.12,3.0,150.0,8.0,0.20,63.0,"HGB — folhas pequenas, mais folhas por árvore.",NaN,NaN
8,hgb_09_l2_forte,HistGradientBoosting,HistGradientBoostingRegressor,0.10,3.0,250.0,22.0,0.50,15.0,HGB — regularização L2 forte contra overfitting.,NaN,NaN
9,hgb_10_muito_raso,HistGradientBoosting,HistGradientBoostingRegressor,0.08,2.0,280.0,28.0,0.10,5.0,HGB — variante extra-rasa (vizinha da melhor r...,NaN,NaN


## Preparar dados (split temporal)

Mesmo particionamento usado em `run_educational_ml_suite()` — garante coerência metodológica.

In [3]:
YEAR_CUTOFF = 2017
RANDOM_STATE = 42

S = prepare_split(year_cutoff=YEAR_CUTOFF)
X_train, y_train = S["X_train"], S["y_train"]
X_test, y_test = S["X_test"], S["y_test"]
num_f, cat_f = S["numeric_features"], S["categorical_features"]
sig_sample = X_train.iloc[: min(5, len(X_train))]

print(f"Treino: {len(X_train)} linhas | anos {S['train_years']}")
print(f"Teste:  {len(X_test)} linhas | anos {S['test_years']}")

Treino: 54 linhas | anos [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
Teste:  20 linhas | anos [2018, 2019, 2021, 2022]


## Loop de treino + registro MLflow

Para **cada rodada** do plano:
1. imprime o **algoritmo** e os **hiperparâmetros**;
2. **treina** o pipeline (`fit`);
3. **avalia** no teste (MAE, RMSE, R²);
4. **registra** parâmetros, métricas e modelo no MLflow (`log_training_run_to_mlflow`).

In [4]:
mlflow = configure_mlflow(experiment_name=MLFLOW_TUNING_EXPERIMENT)
if mlflow is None:
    raise RuntimeError("MLflow desabilitado. Não defina MLFLOW_DISABLED=1.")

results: list[dict] = []

for rodada, cfg in enumerate(EXPERIMENT_PLAN, start=1):
    algorithm = cfg["algorithm"]
    params = cfg["params"]
    run_name = cfg["run_name"]
    sklearn_name = algorithm_display_name(algorithm)

    print("\n" + "=" * 72)
    print(f"RODADA {rodada:02d} | run: {run_name}")
    print(f"ALGORITMO: {algorithm}  ({sklearn_name})")
    print(f"HIPERPARÂMETROS AJUSTADOS: {params}")
    print("=" * 72)

    # --- 1. Montar pipeline com hiperparâmetros desta rodada ---
    pipe = build_pipeline_from_config(
        algorithm,
        num_f,
        cat_f,
        params,
        random_state=RANDOM_STATE,
    )

    # --- 2. Treinar ---
    pipe.fit(X_train, y_train)

    # --- 3. Avaliar no holdout temporal ---
    y_pred = pipe.predict(X_test)
    metrics = evaluate_regression(y_test, y_pred)
    print(
        f"Métricas teste → MAE={metrics['mae']:.3f} | "
        f"RMSE={metrics['rmse']:.3f} | R²={metrics['r2']:.3f}"
    )

    # --- 4. Registrar no MLflow ---
    run_id = log_training_run_to_mlflow(
        mlflow,
        run_name=run_name,
        algorithm=algorithm,
        params=params,
        pipe=pipe,
        metrics=metrics,
        rodada=rodada,
        year_cutoff=YEAR_CUTOFF,
        random_state=RANDOM_STATE,
        sig_sample=sig_sample,
        nota=str(cfg.get("nota", "")),
    )
    print(f"Registrado no MLflow → run_id={run_id}")

    results.append(
        {
            "rodada": rodada,
            "run_name": run_name,
            "run_id": run_id,
            "algorithm": algorithm,
            "modelo_sklearn": sklearn_name,
            **params,
            "test_mae": metrics["mae"],
            "test_rmse": metrics["rmse"],
            "test_r2": metrics["r2"],
        }
    )

summary = pd.DataFrame(results)
out_csv = ROOT / "outputs" / "ml" / "mlflow_campaign_summary.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_csv, index=False)
print(f"\n{len(summary)} runs registrados em '{MLFLOW_TUNING_EXPERIMENT}'")
print(f"Resumo salvo em: {out_csv}")


RODADA 01 | run: hgb_01_baseline
ALGORITMO: HistGradientBoosting  (HistGradientBoostingRegressor)
HIPERPARÂMETROS AJUSTADOS: {'learning_rate': 0.15, 'max_depth': 3, 'max_iter': 80, 'min_samples_leaf': 24, 'l2_regularization': 0.0, 'max_leaf_nodes': 15}
Métricas teste → MAE=0.855 | RMSE=1.306 | R²=0.687


/home/fatima/Henrique_projetos/projeto_evasao_escolar/.venv/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/15 16:19:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/15 16:19:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registrado no MLflow → run_id=8f74fb214d64459b870a4aa200f91f1a

RODADA 02 | run: hgb_02_mais_iter_l2
ALGORITMO: HistGradientBoosting  (HistGradientBoostingRegressor)
HIPERPARÂMETROS AJUSTADOS: {'learning_rate': 0.1, 'max_depth': 4, 'max_iter': 180, 'min_samples_leaf': 16, 'l2_regularization': 0.1, 'max_leaf_nodes': 31}
Métricas teste → MAE=0.717 | RMSE=0.917 | R²=0.846


/home/fatima/Henrique_projetos/projeto_evasao_escolar/.venv/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/15 16:20:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/06/15 16:22:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/home/fatima/Henrique_projetos/projeto_evasao_escolar/.venv/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/15 16:22:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Registrado no MLflow → run_id=a54e42db73de4382a4d1695c4f8c0142

RODADA 30 | run: knn_10_k5_uniform
ALGORITMO: KNeighbors  (KNeighborsRegressor)
HIPERPARÂMETROS AJUSTADOS: {'n_neighbors': 5, 'weights': 'uniform'}
Métricas teste → MAE=1.113 | RMSE=1.961 | R²=0.295


2026/06/15 16:22:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registrado no MLflow → run_id=4eca8855af39475aa7b7dc99577de292

30 runs registrados em 'evasao_treino_parametros'
Resumo salvo em: /home/fatima/Henrique_projetos/projeto_evasao_escolar/outputs/ml/mlflow_campaign_summary.csv


## Resultados por algoritmo (menor MAE = melhor no teste)

In [5]:
for algo in ["HistGradientBoosting", "DecisionTree", "KNeighbors"]:
    sub = summary[summary["algorithm"] == algo].sort_values("test_mae")
    print(f"\n### {algorithm_display_name(algo)}")
    display(sub[["run_name", "test_mae", "test_rmse", "test_r2"] + [
        c for c in sub.columns if c not in (
            "rodada", "run_name", "run_id", "algorithm", "modelo_sklearn",
            "test_mae", "test_rmse", "test_r2",
        )
    ]])

best = summary.sort_values("test_mae").iloc[0]
print(
    f"\nMelhor run global: {best['run_name']} | {best['modelo_sklearn']} | "
    f"MAE={best['test_mae']:.3f}"
)


### HistGradientBoostingRegressor


,run_name,test_mae,test_rmse,test_r2,learning_rate,max_depth,max_iter,min_samples_leaf,l2_regularization,max_leaf_nodes,n_neighbors,weights
2,hgb_03_arvore_rasa,0.546508,0.713425,0.906752,0.10,2.0,220.0,20.0,0.05,7.0,NaN,NaN
5,hgb_06_lr_baixo_longo,0.626015,0.761209,0.893842,0.05,3.0,300.0,18.0,0.10,31.0,NaN,NaN
6,hgb_07_profundo_moderado,0.632469,0.821909,0.876237,0.08,5.0,200.0,14.0,0.15,31.0,NaN,NaN
7,hgb_08_folhas_pequenas,0.658216,0.908298,0.848852,0.12,3.0,150.0,8.0,0.20,63.0,NaN,NaN
1,hgb_02_mais_iter_l2,0.716982,0.916856,0.845991,0.10,4.0,180.0,16.0,0.10,31.0,NaN,NaN
3,hgb_04_balanceado,0.739700,0.989148,0.820747,0.10,4.0,220.0,12.0,0.30,15.0,NaN,NaN
0,hgb_01_baseline,0.854759,1.306494,0.687277,0.15,3.0,80.0,24.0,0.00,15.0,NaN,NaN
8,hgb_09_l2_forte,0.906833,1.379529,0.651337,0.10,3.0,250.0,22.0,0.50,15.0,NaN,NaN
4,hgb_05_lr_alto_curto,4.978333,5.450377,-4.442489,0.20,3.0,60.0,30.0,0.00,15.0,NaN,NaN
9,hgb_10_muito_raso,4.978333,5.450377,-4.442489,0.08,2.0,280.0,28.0,0.10,5.0,NaN,NaN



### DecisionTreeRegressor


,run_name,test_mae,test_rmse,test_r2,learning_rate,max_depth,max_iter,min_samples_leaf,l2_regularization,max_leaf_nodes,n_neighbors,weights
10,tree_01_rasa,0.977517,1.570987,0.547843,NaN,3.0,NaN,12.0,NaN,NaN,NaN,NaN
13,tree_04_folhas_grandes,1.041908,1.718424,0.458990,NaN,5.0,NaN,16.0,NaN,NaN,NaN,NaN
17,tree_08_intermediaria,1.225833,2.119254,0.177169,NaN,5.0,NaN,6.0,NaN,NaN,NaN,NaN
11,tree_02_media,1.360909,2.225345,0.092724,NaN,4.0,NaN,8.0,NaN,NaN,NaN,NaN
16,tree_07_profunda_restrita,1.360909,2.225345,0.092724,NaN,7.0,NaN,8.0,NaN,NaN,NaN,NaN
12,tree_03_profunda,1.497500,2.063023,0.220255,NaN,6.0,NaN,4.0,NaN,NaN,NaN,NaN
19,tree_10_muito_profunda,1.552000,2.148972,0.153931,NaN,8.0,NaN,5.0,NaN,NaN,NaN,NaN
15,tree_06_media_folhas_grandes,2.146023,3.928342,-1.827243,NaN,4.0,NaN,20.0,NaN,NaN,NaN,NaN
14,tree_05_muito_rasa,2.146023,3.928342,-1.827243,NaN,2.0,NaN,20.0,NaN,NaN,NaN,NaN
18,tree_09_super_rasa,4.978333,5.450377,-4.442489,NaN,2.0,NaN,30.0,NaN,NaN,NaN,NaN



### KNeighborsRegressor


,run_name,test_mae,test_rmse,test_r2,learning_rate,max_depth,max_iter,min_samples_leaf,l2_regularization,max_leaf_nodes,n_neighbors,weights
20,knn_01_poucos_vizinhos,0.828333,1.366890,0.657696,NaN,NaN,NaN,NaN,NaN,NaN,3.0,uniform
21,knn_02_padrao,0.854601,1.474601,0.601624,NaN,NaN,NaN,NaN,NaN,NaN,5.0,distance
25,knn_06_k6_distance,0.925538,1.615223,0.522020,NaN,NaN,NaN,NaN,NaN,NaN,6.0,distance
22,knn_03_mais_vizinhos,0.967718,1.691099,0.476058,NaN,NaN,NaN,NaN,NaN,NaN,7.0,distance
24,knn_05_k4_uniform,0.997500,1.727462,0.453284,NaN,NaN,NaN,NaN,NaN,NaN,4.0,uniform
26,knn_07_k9_distance,1.041239,1.829558,0.386751,NaN,NaN,NaN,NaN,NaN,NaN,9.0,distance
29,knn_10_k5_uniform,1.113000,1.961097,0.295400,NaN,NaN,NaN,NaN,NaN,NaN,5.0,uniform
23,knn_04_muitos_vizinhos,1.119432,1.926203,0.320251,NaN,NaN,NaN,NaN,NaN,NaN,11.0,distance
27,knn_08_k13_distance,1.171847,1.989426,0.274897,NaN,NaN,NaN,NaN,NaN,NaN,13.0,distance
28,knn_09_k15_uniform,1.433333,2.408758,-0.062994,NaN,NaN,NaN,NaN,NaN,NaN,15.0,uniform



Melhor run global: hgb_03_arvore_rasa | HistGradientBoostingRegressor | MAE=0.547
